# Sparsity patterns and derivative products
The prescribed and the supports' patterns of the five-cell zone and the 64-cell circle: the medians of the AD products over the control periods (the products of the counts table), the products at control period 233 of the circle (the products against the horizon), and the pattern figure. CPU, float64; the paper's values are read from `results/`.

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")  # MUMPS and its BLAS on one thread, as the paper's solves ran
import jax
import numpy as np

jax.config.update("jax_enable_x64", True)
import time

from common import build, counts, paper
from sparsemax_dstl.sparse import counts as SC
from sparsemax_dstl.tasks import dual_arm_scenes as S

Medians over every tenth control period and the next (on the circle, those whose horizon ends by sample 999); the Lagrangian Hessian at every hundredth period of the five-cell zone and at periods 0 and 300 of the circle.

In [2]:
zone_100 = counts("five-cell zone", 100)
zone_200 = counts("five-cell zone", 200)

five-cell zone T 100 | 200 periods, 10 Hessian periods | 58.9 s
   constraint Jacobian products, dense / prescribed / supports: [2806.0, 7.0, 4.0] | paper [2806.0, 7.0, 4.0]
   Hessian of the Lagrangian products, dense / prescribed / supports: [4200.0, 1011.5, 59.5] | paper [4200.0, 1011.5, 59.5]


five-cell zone T 200 | 200 periods, 10 Hessian periods | 130.6 s
   constraint Jacobian products, dense / prescribed / supports: [5612.0, 8.0, 4.0] | paper [5612.0, 8.0, 4.0]
   Hessian of the Lagrangian products, dense / prescribed / supports: [8400.0, 1204.0, 63.0] | paper [8400.0, 1204.0, 63.0]


In [3]:
circle_100 = counts("64-cell circle", 100)
circle_200 = counts("64-cell circle", 200)

64-cell circle T 100 | 180 periods, 2 Hessian periods | 151.1 s
   constraint Jacobian products, dense / prescribed / supports: [2823.0, 25.0, 9.0] | paper [2823.0, 25.0, 9.0]
   Hessian of the Lagrangian products, dense / prescribed / supports: [4200.0, 1393.0, 203.0] | paper [4200.0, 1393.0, 203.0]


64-cell circle T 200 | 160 periods, 2 Hessian periods | 385.2 s
   constraint Jacobian products, dense / prescribed / supports: [5713.0, 95.5, 13.0] | paper [5713.0, 95.5, 13.0]
   Hessian of the Lagrangian products, dense / prescribed / supports: [8400.0, 2086.0, 266.0] | paper [8400.0, 2086.0, 266.0]


The circle's constraint Jacobian at control period 233, the period of the derivative times: one reverse product per constraint against the row coloring of the supports' pattern.

In [4]:
for T in (100, 200):
    scene, conjs = build(S.circle(T))
    A, meta = SC.count(scene, conjs, np.array([233]), np.array([], int))
    dense, supports = meta["n"] + int(A["all_rows"][0]), int(A["whole_supports_products"][0])
    m = dict(scene="64-cell circle", horizon=T)
    print("T", T, "| dense", dense, "supports", supports, "| paper", paper("products_horizon", comparator="dense AD", **m),
          paper("products_horizon", comparator="supports' pattern", **m))

T 100 | dense 2830 supports 10 | paper 2830.0 10.0


T 200 | dense 5723 supports 12 | paper 5723.0 12.0


The figure's two periods: the circle's constraint Jacobian at 200 time steps, control period 50, and the five-cell zone's Lagrangian Hessian at 400 time steps, control period 0.

In [5]:
scene, conjs = build(S.circle(200))
jac, _ = SC.count(scene, conjs, np.array([50, 51]), np.array([], int), figure_period=50)
t = time.perf_counter()
scene, conjs = build(S.five_cell_zone(400))
hess, _ = SC.count(scene, conjs, np.array([0, 1]), np.array([0]), figure_period=0)
print("zone, 400 time steps, one Hessian period:", round(time.perf_counter() - t, 1), "s")

zone, 400 time steps, one Hessian period: 36.7 s


The figure's arrays, saved to `out/patterns.npz`: the circle's specification rows binned by row, arm and time step (no row reads a velocity or an acceleration) and the zone's upper triangle binned by pairs of time steps, with the entries and products the figure prints.

In [6]:
z = {"jacobian_T": 200, "jacobian_period": 50, "jacobian_row_kind": jac["figure_row_kind"], "hessian_T": 400, "hessian_period": 0}
for lev, name in (("prescribed", "prescribed pattern"), ("supports", "supports' pattern")):
    r, c = jac["figure_jacobian_" + lev + "_rows"], jac["figure_jacobian_" + lev + "_cols"]
    assert ((c % 28 < 14) & (c < 28 * 200)).all()
    cells = np.zeros((z["jacobian_row_kind"].size, 2, 200), np.int8)
    np.add.at(cells, (r, (c % 14) // 7, c // 28), 1)
    z["jacobian_cells_" + lev] = cells
    z["jacobian_entries_" + lev] = int(jac["all_" + lev + "_nnz"][0])
    z["jacobian_products_" + lev] = int(jac["whole_" + lev + "_products"][0])
    r, c = hess["figure_hessian_" + lev + "_rows"], hess["figure_hessian_" + lev + "_cols"]
    up = r <= c
    cells = np.zeros((400, 400), np.int16)
    np.add.at(cells, (r[up] // 28, c[up] // 28), 1)
    z["hessian_cells_" + lev] = cells
    z["hessian_upper_" + lev] = int(hess["hessian_all_" + lev + "_upper"][0])
    z["hessian_products_" + lev] = int(hess["hessian_whole_" + lev + "_star"][0])
    mj, mh = dict(scene="64-cell circle", horizon=200, comparator=name), dict(scene="five-cell zone", horizon=400, comparator=name)
    print(name, "| Jacobian entries", z["jacobian_entries_" + lev], "products", z["jacobian_products_" + lev], "| paper",
          paper("patterns", quantity="constraint Jacobian entries of the specification rows", **mj), paper("patterns", quantity="constraint Jacobian products", **mj))
    print(name, "| Hessian upper entries", z["hessian_upper_" + lev], "products", z["hessian_products_" + lev], "| paper",
          paper("patterns", quantity="Hessian of the Lagrangian upper entries of the joint positions", **mh), paper("patterns", quantity="Hessian of the Lagrangian products", **mh))
os.makedirs("out", exist_ok=True)
np.savez_compressed("out/patterns.npz", **z)

prescribed pattern | Jacobian entries 146398 products 95 | paper 146398.0 95.0
prescribed pattern | Hessian upper entries 2944298 products 1204 | paper 2944298.0 1204.0
supports' pattern | Jacobian entries 7392 products 13 | paper 7392.0 13.0
supports' pattern | Hessian upper entries 27363 products 63 | paper 27363.0 63.0


The pattern figure (`out/jacobian-patterns` and `out/hessian-patterns`, SVG and PDF) and the products against the horizon from `results/products_horizon.csv` (`out/products-horizon`).

In [7]:
%run figures/jacobian_patterns.py

period 50 T 200 rows {'until': 107, 'reach': 2}
prescribed pattern: 146,398 entries, 95 AD products
supports' pattern (SparseMax): 7,392 entries, 13 AD products


In [8]:
%run figures/hessian_patterns.py

period 0 T 400 enlarged time steps 3 to 42
('prescribed pattern', '2,944,298 entries', '1,204 products')
('supports (SparseMax)', '27,363 entries', '63 products')


In [9]:
%run figures/products_horizon.py

Hessian [100, 200, 400] dense [4200.0, 8400.0, 16800.0] prescribed [1011.5, 1204.0, 1204.0] supports [59.5, 63.0, 63.0]
Jacobian [100, 200, 400] dense [2830.0, 5723.0, 11500.0] CasADi [32.0, 98.0, 130.0] supports [10.0, 12.0, 13.0]
